# 🌙 SleepSense AI — Sleep Stage Classification
### Training Pipeline with Visual Result Matrices

Trains a **Random Forest classifier** on polysomnography data (`ACC_X`, `ACC_Y`, `ACC_Z`, `HR`)  
to predict sleep stages: **P, W, N1, N2, N3, R**

> Upload your CSV files (`compressed_S002_whole_df.csv`, `compressed_S006_whole_df.csv`) before running.

---
## 📦 Step 1 — Install & Import Dependencies

In [ ]:
!pip install -q joblib scikit-learn matplotlib seaborn pandas numpy

In [ ]:
from __future__ import annotations
import json, time, warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    roc_curve, auc, precision_recall_curve, average_precision_score,
    precision_score, recall_score, f1_score
)
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.preprocessing import LabelEncoder, label_binarize

warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.facecolor": "#0d1117",
    "axes.facecolor":   "#161b22",
    "axes.edgecolor":   "#30363d",
    "axes.labelcolor":  "#e6edf3",
    "xtick.color":      "#8b949e",
    "ytick.color":      "#8b949e",
    "text.color":       "#e6edf3",
    "grid.color":       "#21262d",
    "grid.linewidth":   0.8,
    "font.family":      "monospace",
    "axes.titlesize":   13,
    "axes.labelsize":   11,
})

STAGE_COLORS = {
    "P":  "#6366f1",
    "W":  "#f59e0b",
    "N1": "#22d3ee",
    "N2": "#3b82f6",
    "N3": "#8b5cf6",
    "R":  "#ef4444",
}
print("✅ Imports complete")

---
## 📁 Step 2 — Upload Data Files

In [ ]:
import os
IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    from google.colab import files
    print("📤 Upload compressed_S002_whole_df.csv and compressed_S006_whole_df.csv")
    uploaded = files.upload()
    DATA_DIR = Path(".")
else:
    DATA_DIR = Path(".")   # change to your data directory if needed
    print(f"📂 Data directory: {DATA_DIR.resolve()}")

MODEL_PATH  = DATA_DIR / "sleep_model.joblib"
FEATURE_CFG = DATA_DIR / "feature_config.json"
DATASETS     = ["compressed_S002_whole_df.csv", "compressed_S006_whole_df.csv"]
WINDOW_SIZE  = 64
VALID_STAGES = {"P", "W", "N1", "N2", "N3", "R"}
RANDOM_STATE = 42
STAGE_LABELS = {0: "P", 1: "W", 2: "N1", 3: "N2", 4: "N3", 5: "R"}
print("✅ Config ready")

---
## 🗄️ Step 3 — Load & Clean Data

In [ ]:
def load_and_clean(filepath):
    print(f"  Loading {filepath.name}...", end=" ")
    df = pd.read_csv(filepath, low_memory=False)
    df = df[[c for c in df.columns if not c.startswith("Unnamed")]]
    df = df[df["Sleep_Stage"] != "Sleep_Stage"]
    df = df[df["Sleep_Stage"].isin(VALID_STAGES)]
    for col in ["TIMESTAMP", "ACC_X", "ACC_Y", "ACC_Z", "HR"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=["ACC_X", "ACC_Y", "ACC_Z", "HR"]).reset_index(drop=True)
    print(f"{len(df):,} rows | stages: {df['Sleep_Stage'].value_counts().to_dict()}")
    return df

dfs = []
for fname in DATASETS:
    fpath = DATA_DIR / fname
    if fpath.exists():
        dfs.append(load_and_clean(fpath))
    else:
        print(f"  ⚠️  {fname} not found — skipping")

if not dfs:
    raise FileNotFoundError("No dataset files found. Upload them first.")

combined = pd.concat(dfs, ignore_index=True)
print(f"\n📊 Combined: {len(combined):,} rows")
print(combined["Sleep_Stage"].value_counts())

---
## 📊 Step 4 — Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("EDA — Raw Data Overview", fontsize=15, color="#e6edf3", fontweight="bold", y=1.02)

# Stage distribution
ax = axes[0]
stage_counts = combined["Sleep_Stage"].value_counts().reindex(["P","W","N1","N2","N3","R"], fill_value=0)
bars = ax.bar(stage_counts.index, stage_counts.values,
              color=[STAGE_COLORS[s] for s in stage_counts.index], edgecolor="none", alpha=0.9, width=0.6)
ax.set_title("Stage Class Distribution", pad=10)
ax.set_xlabel("Sleep Stage"); ax.set_ylabel("Sample Count"); ax.grid(axis="y", alpha=0.4)
for bar, count in zip(bars, stage_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(stage_counts)*0.01,
            f"{count:,}", ha="center", va="bottom", fontsize=9, color="#8b949e")

# HR distribution
ax = axes[1]
for stage in ["P","W","N1","N2","N3","R"]:
    data = combined[combined["Sleep_Stage"] == stage]["HR"].dropna()
    if len(data) > 0:
        ax.hist(data, bins=40, alpha=0.55, color=STAGE_COLORS[stage], label=stage, density=True, edgecolor="none")
ax.set_title("HR Distribution by Stage", pad=10)
ax.set_xlabel("Heart Rate (bpm)"); ax.set_ylabel("Density")
ax.legend(fontsize=9, framealpha=0.3); ax.grid(alpha=0.4)

# Acc magnitude box plot
ax = axes[2]
combined["acc_mag"] = np.sqrt(combined["ACC_X"]**2 + combined["ACC_Y"]**2 + combined["ACC_Z"]**2)
stage_order = ["P","W","N1","N2","N3","R"]
data_by_stage = [combined[combined["Sleep_Stage"]==s]["acc_mag"].dropna().values for s in stage_order]
bp = ax.boxplot(data_by_stage, patch_artist=True, vert=True,
                medianprops=dict(color="white", linewidth=2),
                whiskerprops=dict(color="#8b949e"), capprops=dict(color="#8b949e"),
                flierprops=dict(marker="o", markersize=2, alpha=0.3, color="#8b949e"))
for patch, stage in zip(bp["boxes"], stage_order):
    patch.set_facecolor(STAGE_COLORS[stage]); patch.set_alpha(0.75)
ax.set_xticklabels(stage_order)
ax.set_title("Acc Magnitude by Stage", pad=10)
ax.set_xlabel("Sleep Stage"); ax.set_ylabel("Acceleration Magnitude"); ax.grid(axis="y", alpha=0.4)

plt.tight_layout()
plt.savefig("eda_overview.png", dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show()
print("✅ Saved eda_overview.png")

---
## ⚙️ Step 5 — Feature Engineering

In [ ]:
def engineer_features(df):
    print("  Engineering features...")
    feat = pd.DataFrame(index=df.index)
    feat["acc_x"] = df["ACC_X"].values
    feat["acc_y"] = df["ACC_Y"].values
    feat["acc_z"] = df["ACC_Z"].values
    feat["hr"]    = df["HR"].values
    feat["acc_mag"] = np.sqrt(
        df["ACC_X"].astype(float)**2 +
        df["ACC_Y"].astype(float)**2 +
        df["ACC_Z"].astype(float)**2
    )
    window = WINDOW_SIZE
    for col in ["acc_x", "acc_y", "acc_z", "hr", "acc_mag"]:
        series = feat[col]
        feat[f"{col}_roll_mean"] = series.rolling(window, min_periods=1).mean()
        feat[f"{col}_roll_std"]  = series.rolling(window, min_periods=1).std().fillna(0)
        feat[f"{col}_roll_min"]  = series.rolling(window, min_periods=1).min()
        feat[f"{col}_roll_max"]  = series.rolling(window, min_periods=1).max()
    feat["movement_intensity"] = feat["acc_mag"].rolling(window, min_periods=1).std().fillna(0)
    feat["hr_delta"] = feat["hr"].diff().fillna(0)
    feat["hr_delta_roll_mean"] = feat["hr_delta"].rolling(window, min_periods=1).mean()
    for axis in ["acc_x", "acc_y", "acc_z"]:
        feat[f"{axis}_roll_range"] = feat[f"{axis}_roll_max"] - feat[f"{axis}_roll_min"]
    for lag in [1, 2, 3, 5, 10]:
        feat[f"acc_mag_lag{lag}"] = feat["acc_mag"].shift(lag).fillna(feat["acc_mag"].iloc[0])
        feat[f"hr_lag{lag}"]     = feat["hr"].shift(lag).fillna(feat["hr"].iloc[0])
    for axis in ["acc_x", "acc_y", "acc_z"]:
        sign_changes = (feat[axis].diff().fillna(0) != 0).astype(int)
        feat[f"{axis}_zcr"] = sign_changes.rolling(window, min_periods=1).sum()
    print(f"    → {feat.shape[1]} features generated")
    return feat

features = engineer_features(combined)
labels   = combined["Sleep_Stage"].values
le = LabelEncoder()
le.fit(sorted(VALID_STAGES))
y = le.transform(labels)
feature_names = features.columns.tolist()
X = features.values.astype(np.float32)
print(f"\n✅ Feature matrix: {X.shape}")
print(f"   Classes: {le.classes_.tolist()}")

---
## 🏋️ Step 6 — Train/Test Split & Model Training

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(f"Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")

print("\n🌲 Training Random Forest...")
t0 = time.time()
model = RandomForestClassifier(
    n_estimators=200, max_depth=25, min_samples_split=10,
    min_samples_leaf=4, class_weight="balanced",
    n_jobs=-1, random_state=RANDOM_STATE, verbose=0,
)
model.fit(X_train, y_train)
print(f"✅ Training complete in {time.time() - t0:.1f}s")

y_pred      = model.predict(X_test)
y_pred_prob = model.predict_proba(X_test)
acc         = accuracy_score(y_test, y_pred)
target_names = le.classes_.tolist()
print(f"\n🎯 Overall Accuracy: {acc:.4f} ({acc*100:.1f}%)")

---
## 📉 Visualization 1 — Confusion Matrix (Raw + Normalized)

In [ ]:
def plot_confusion_matrix(y_test, y_pred, target_names):
    cm = confusion_matrix(y_test, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis]
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle("Confusion Matrix", fontsize=15, color="#e6edf3", fontweight="bold")
    cmap = LinearSegmentedColormap.from_list("sleep", ["#161b22", "#3b82f6", "#6366f1"])
    for ax, data, title, fmt in zip(
        axes, [cm, cm_norm], ["Raw Counts", "Normalized (Recall)"], ["d", ".2f"]
    ):
        sns.heatmap(data, annot=True, fmt=fmt, cmap=cmap,
                    xticklabels=target_names, yticklabels=target_names,
                    linewidths=0.5, linecolor="#0d1117", ax=ax, cbar=True,
                    annot_kws={"size": 11, "weight": "bold"})
        ax.set_title(title, pad=10)
        ax.set_xlabel("Predicted Stage"); ax.set_ylabel("True Stage")
        for i in range(len(target_names)):
            ax.add_patch(plt.Rectangle((i, i), 1, 1, fill=False,
                         edgecolor=STAGE_COLORS[target_names[i]], lw=2.5))
    plt.tight_layout()
    plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight", facecolor="#0d1117")
    plt.show()
    print("✅ Saved confusion_matrix.png")

plot_confusion_matrix(y_test, y_pred, target_names)

---
## 📋 Visualization 2 — Per-Class Metric Heatmap & Bar Chart

In [ ]:
def plot_classification_report(y_test, y_pred, target_names):
    p = precision_score(y_test, y_pred, average=None)
    r = recall_score(y_test, y_pred, average=None)
    f = f1_score(y_test, y_pred, average=None)
    metrics_df = pd.DataFrame({"Precision": p, "Recall": r, "F1-Score": f}, index=target_names)

    fig, axes = plt.subplots(1, 2, figsize=(17, 5))
    fig.suptitle("Per-Class Classification Metrics", fontsize=15, color="#e6edf3", fontweight="bold")

    cmap2 = LinearSegmentedColormap.from_list("green", ["#161b22", "#1f6b3c", "#22c55e"])
    sns.heatmap(metrics_df, annot=True, fmt=".3f", cmap=cmap2,
                linewidths=0.5, linecolor="#0d1117", vmin=0, vmax=1,
                ax=axes[0], annot_kws={"size": 12, "weight": "bold"})
    axes[0].set_title("Precision / Recall / F1 per Stage", pad=10)
    axes[0].set_ylabel("Sleep Stage")

    ax = axes[1]
    x = np.arange(len(target_names)); width = 0.25
    b1 = ax.bar(x - width, p, width, label="Precision", color="#3b82f6", alpha=0.9)
    b2 = ax.bar(x,          r, width, label="Recall",    color="#22c55e", alpha=0.9)
    b3 = ax.bar(x + width,  f, width, label="F1-Score",  color="#f59e0b", alpha=0.9)
    ax.set_xticks(x); ax.set_xticklabels(target_names)
    ax.set_ylim(0, 1.15); ax.set_title("Metric Comparison per Stage", pad=10)
    ax.set_xlabel("Sleep Stage"); ax.set_ylabel("Score")
    ax.legend(fontsize=10, framealpha=0.3); ax.grid(axis="y", alpha=0.4)
    for bars in [b1, b2, b3]:
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.01,
                    f"{h:.2f}", ha="center", va="bottom", fontsize=7.5, color="#8b949e")
    plt.tight_layout()
    plt.savefig("per_class_metrics.png", dpi=150, bbox_inches="tight", facecolor="#0d1117")
    plt.show()
    print("✅ Saved per_class_metrics.png")

plot_classification_report(y_test, y_pred, target_names)

---
## 📈 Visualization 3 — Multi-Class ROC & Precision-Recall Curves

In [ ]:
def plot_roc_and_pr_curves(y_test, y_pred_prob, target_names):
    n_classes = len(target_names)
    y_bin = label_binarize(y_test, classes=list(range(n_classes)))
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle("ROC & Precision-Recall Curves (One-vs-Rest)", fontsize=15, color="#e6edf3", fontweight="bold")

    ax = axes[0]
    ax.plot([0,1], [0,1], "w--", lw=1, alpha=0.4, label="Random (AUC=0.50)")
    for i, stage in enumerate(target_names):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_pred_prob[:, i])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=STAGE_COLORS[stage], lw=2, label=f"{stage}  (AUC={roc_auc:.3f})")
    ax.set_title("ROC Curves", pad=10); ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.legend(fontsize=9.5, framealpha=0.3, loc="lower right")
    ax.grid(alpha=0.35); ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])

    ax = axes[1]
    for i, stage in enumerate(target_names):
        prec, rec, _ = precision_recall_curve(y_bin[:, i], y_pred_prob[:, i])
        ap = average_precision_score(y_bin[:, i], y_pred_prob[:, i])
        ax.plot(rec, prec, color=STAGE_COLORS[stage], lw=2, label=f"{stage}  (AP={ap:.3f})")
    ax.set_title("Precision-Recall Curves", pad=10); ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.legend(fontsize=9.5, framealpha=0.3, loc="lower left")
    ax.grid(alpha=0.35); ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])

    plt.tight_layout()
    plt.savefig("roc_pr_curves.png", dpi=150, bbox_inches="tight", facecolor="#0d1117")
    plt.show()
    print("✅ Saved roc_pr_curves.png")

plot_roc_and_pr_curves(y_test, y_pred_prob, target_names)

---
## 🌟 Visualization 4 — Feature Importance (Top 25 + Error Bars)

In [ ]:
def plot_feature_importance(model, feature_names, top_n=25):
    importances = model.feature_importances_
    std = np.std([tree.feature_importances_ for tree in model.estimators_], axis=0)
    sorted_idx = np.argsort(importances)[::-1][:top_n]
    top_names = [feature_names[i] for i in sorted_idx]
    top_imp   = importances[sorted_idx]
    top_std   = std[sorted_idx]

    def feature_color(name):
        if "hr" in name:       return "#ef4444"
        if "acc_mag" in name:  return "#8b5cf6"
        if "acc_x" in name:    return "#22d3ee"
        if "acc_y" in name:    return "#3b82f6"
        if "acc_z" in name:    return "#6366f1"
        if "movement" in name: return "#f59e0b"
        return "#8b949e"

    colors = [feature_color(n) for n in top_names]
    fig, ax = plt.subplots(figsize=(13, 9))
    fig.suptitle(f"Top {top_n} Feature Importances (Mean Decrease in Impurity)",
                 fontsize=14, color="#e6edf3", fontweight="bold")
    y_pos = np.arange(len(top_names))
    ax.barh(y_pos, top_imp[::-1], xerr=top_std[::-1], color=colors[::-1], alpha=0.85, edgecolor="none",
            error_kw=dict(ecolor="#8b949e", capsize=3, lw=1))
    ax.set_yticks(y_pos); ax.set_yticklabels(top_names[::-1], fontsize=10)
    ax.set_xlabel("Feature Importance (MDI)"); ax.grid(axis="x", alpha=0.35)
    legend_items = [
        mpatches.Patch(color="#ef4444", label="Heart Rate"),
        mpatches.Patch(color="#8b5cf6", label="Acc Magnitude"),
        mpatches.Patch(color="#22d3ee", label="ACC X"),
        mpatches.Patch(color="#3b82f6", label="ACC Y"),
        mpatches.Patch(color="#6366f1", label="ACC Z"),
        mpatches.Patch(color="#f59e0b", label="Movement"),
    ]
    ax.legend(handles=legend_items, loc="lower right", fontsize=9, framealpha=0.3)
    plt.tight_layout()
    plt.savefig("feature_importance.png", dpi=150, bbox_inches="tight", facecolor="#0d1117")
    plt.show()
    print("✅ Saved feature_importance.png")

plot_feature_importance(model, feature_names, top_n=25)

---
## 📉 Visualization 5 — Learning Curve

In [ ]:
def plot_learning_curve(model, X, y):
    print("Computing learning curve (may take a minute)...")
    train_sizes, train_scores, val_scores = learning_curve(
        model, X, y,
        train_sizes=np.linspace(0.1, 1.0, 8),
        cv=3, scoring="accuracy", n_jobs=-1,
    )
    tm = np.mean(train_scores, axis=1); ts = np.std(train_scores, axis=1)
    vm = np.mean(val_scores, axis=1);   vs = np.std(val_scores, axis=1)
    fig, ax = plt.subplots(figsize=(11, 5))
    fig.suptitle("Learning Curve — Accuracy vs Training Size",
                 fontsize=14, color="#e6edf3", fontweight="bold")
    ax.fill_between(train_sizes, tm - ts, tm + ts, alpha=0.15, color="#3b82f6")
    ax.fill_between(train_sizes, vm - vs, vm + vs, alpha=0.15, color="#22c55e")
    ax.plot(train_sizes, tm, "o-", color="#3b82f6", lw=2, label="Training Accuracy")
    ax.plot(train_sizes, vm, "s-", color="#22c55e", lw=2, label="Validation Accuracy")
    ax.set_xlabel("Training Samples"); ax.set_ylabel("Accuracy")
    ax.set_ylim(0, 1.05); ax.legend(fontsize=11, framealpha=0.3); ax.grid(alpha=0.4)
    plt.tight_layout()
    plt.savefig("learning_curve.png", dpi=150, bbox_inches="tight", facecolor="#0d1117")
    plt.show()
    print("✅ Saved learning_curve.png")

plot_learning_curve(model, X, y)

---
## 🏁 Visualization 6 — Model Summary Dashboard

In [ ]:
def plot_summary_dashboard(y_test, y_pred, y_pred_prob, target_names, acc, model, feature_names):
    fig = plt.figure(figsize=(20, 12))
    fig.patch.set_facecolor("#0d1117")
    gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.45, wspace=0.35)
    fig.suptitle("SleepSense AI — Model Summary Dashboard",
                 fontsize=17, color="#e6edf3", fontweight="bold", y=1.01)
    sc = [STAGE_COLORS[s] for s in target_names]
    p = precision_score(y_test, y_pred, average=None)
    r = recall_score(y_test, y_pred, average=None)
    f = f1_score(y_test, y_pred, average=None)

    # Accuracy gauge
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.set_facecolor("#161b22")
    theta = np.linspace(0, 2*np.pi*acc, 300)
    ax1.plot(np.cos(theta), np.sin(theta), lw=8, color="#22c55e", solid_capstyle="round")
    ax1.add_patch(plt.Circle((0,0), 0.82, color="#0d1117", zorder=5))
    ax1.text(0, 0.1, f"{acc*100:.1f}%", ha="center", fontsize=24, fontweight="bold", color="#22c55e", zorder=6)
    ax1.text(0, -0.28, "Accuracy", ha="center", fontsize=11, color="#8b949e", zorder=6)
    ax1.set_xlim(-1.2, 1.2); ax1.set_ylim(-1.2, 1.2)
    ax1.set_aspect("equal"); ax1.axis("off"); ax1.set_title("Overall Accuracy", pad=8)

    # F1 bars
    ax2 = fig.add_subplot(gs[0, 1])
    bars = ax2.bar(target_names, f, color=sc, alpha=0.85, width=0.6, edgecolor="none")
    ax2.set_ylim(0, 1.15); ax2.set_title("F1 Score per Stage", pad=8)
    ax2.set_xlabel("Stage"); ax2.set_ylabel("F1"); ax2.grid(axis="y", alpha=0.35)
    for bar, v in zip(bars, f):
        ax2.text(bar.get_x() + bar.get_width()/2, v + 0.02, f"{v:.2f}",
                 ha="center", fontsize=9, color="#e6edf3")

    # Pie chart
    ax3 = fig.add_subplot(gs[0, 2])
    support = np.array([np.sum(y_test == i) for i in range(len(target_names))])
    wedges, texts, autotexts = ax3.pie(
        support, labels=target_names, colors=sc,
        autopct="%1.1f%%", startangle=90,
        wedgeprops=dict(edgecolor="#0d1117", linewidth=1.5),
        textprops={"color": "#e6edf3", "fontsize": 10},
    )
    for at in autotexts:
        at.set_fontsize(9); at.set_color("#0d1117")
    ax3.set_title("Test Set Stage Distribution", pad=8)

    # Top 10 features
    ax4 = fig.add_subplot(gs[0, 3])
    importances = model.feature_importances_
    top10_idx  = np.argsort(importances)[::-1][:10]
    top10_names = [feature_names[i][:18] for i in top10_idx]
    top10_vals  = importances[top10_idx]
    y_pos = np.arange(10)
    ax4.barh(y_pos, top10_vals[::-1], color="#6366f1", alpha=0.8, edgecolor="none")
    ax4.set_yticks(y_pos); ax4.set_yticklabels(top10_names[::-1], fontsize=8)
    ax4.set_title("Top 10 Features", pad=8); ax4.set_xlabel("Importance"); ax4.grid(axis="x", alpha=0.35)

    # Confusion matrix
    ax5 = fig.add_subplot(gs[1, :2])
    cm = confusion_matrix(y_test, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis]
    cmap = LinearSegmentedColormap.from_list("sleep", ["#161b22", "#3b82f6", "#6366f1"])
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap=cmap,
                xticklabels=target_names, yticklabels=target_names,
                linewidths=0.5, linecolor="#0d1117", ax=ax5, annot_kws={"size": 10, "weight": "bold"})
    for i in range(len(target_names)):
        ax5.add_patch(plt.Rectangle((i, i), 1, 1, fill=False,
                      edgecolor=STAGE_COLORS[target_names[i]], lw=2.5))
    ax5.set_title("Normalized Confusion Matrix", pad=8)
    ax5.set_xlabel("Predicted"); ax5.set_ylabel("True")

    # Precision vs Recall scatter
    ax6 = fig.add_subplot(gs[1, 2:])
    for i, stage in enumerate(target_names):
        ax6.scatter(r[i], p[i], s=200, color=STAGE_COLORS[stage], zorder=5, edgecolors="white", linewidths=1)
        ax6.annotate(f"  {stage}\nF1={f[i]:.2f}", (r[i], p[i]),
                     fontsize=9, color=STAGE_COLORS[stage], va="bottom")
    ax6.set_xlim(0, 1.1); ax6.set_ylim(0, 1.1)
    ax6.set_xlabel("Recall"); ax6.set_ylabel("Precision")
    ax6.set_title("Precision vs Recall (per Stage)", pad=8)
    ax6.grid(alpha=0.35)
    ax6.axline((0,0), (1,1), color="white", lw=1, linestyle="--", alpha=0.3)

    plt.savefig("summary_dashboard.png", dpi=150, bbox_inches="tight", facecolor="#0d1117")
    plt.show()
    print("✅ Saved summary_dashboard.png")

plot_summary_dashboard(y_test, y_pred, y_pred_prob, target_names, acc, model, feature_names)

---
## 💾 Step 7 — Save Model & Download Outputs

In [ ]:
artifact = {
    "model":         model,
    "label_encoder": le,
    "feature_names": feature_names,
    "window_size":   WINDOW_SIZE,
    "valid_stages":  sorted(list(VALID_STAGES)),
    "accuracy":      float(acc),
}
joblib.dump(artifact, MODEL_PATH)
print(f"✅ Model saved → {MODEL_PATH}  ({MODEL_PATH.stat().st_size / 1024 / 1024:.1f} MB)")

feature_config = {
    "feature_names": feature_names,
    "window_size":   WINDOW_SIZE,
    "raw_columns":   ["ACC_X", "ACC_Y", "ACC_Z", "HR"],
    "stage_labels":  {int(k): v for k, v in STAGE_LABELS.items()},
    "stage_colors":  STAGE_COLORS,
}
with open(FEATURE_CFG, "w") as f:
    json.dump(feature_config, f, indent=2)
print(f"✅ Feature config saved → {FEATURE_CFG}")

if IN_COLAB:
    from google.colab import files
    print("\n📥 Downloading output files...")
    for fname in ["sleep_model.joblib", "feature_config.json",
                  "summary_dashboard.png", "confusion_matrix.png",
                  "per_class_metrics.png", "roc_pr_curves.png",
                  "feature_importance.png", "learning_curve.png",
                  "eda_overview.png"]:
        if Path(fname).exists():
            files.download(fname)
            print(f"  ⬇️  {fname}")

print("\n🎉 All done!")